# Teach Qwen3-VL to turn pictures into Unity scenes

This fine-tunes **Qwen3-VL** (the Qwen that can see) on pictures made by `make_dataset.py`, so it answers a picture
with the scene JSON that `SceneFromJson.cs` builds in Unity. It trains a LoRA (a small add-on) on a 4-bit model with
[Unsloth](https://github.com/unslothai/unsloth), which fits Colab's free T4 GPU.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. On your PC: `python make_dataset.py --sprites your_sprites --count 4000 --out scenes_dataset`, then zip the
   `scenes_dataset` folder and upload `scenes_dataset.zip` to the top of your Google Drive.
3. Run the cells in order. Training 4,000 pictures for one pass takes roughly 1–2 hours on a T4; checkpoints are saved to
   Drive, so a disconnect doesn't lose the work (set `RESUME = True` and run again).

In [ ]:
%%capture
!pip install unsloth

In [ ]:
MODEL = "unsloth/Qwen3-VL-4B-Instruct"   # 2B-Instruct: faster and smaller; 8B-Instruct: better, but too big for a 6 GB card
ZIP = "/content/drive/MyDrive/scenes_dataset.zip"
OUT = "/content/drive/MyDrive/scenes_model"   # checkpoints and the finished model go here
EPOCHS = 1               # passes over the pictures; 2 helps if you have few pictures (under 2,000)
LORA_RANK = 16           # how big the add-on is; 32 learns more, needs more memory
CHECK_PICTURES = 10      # how many held-back pictures to score before and after training (each takes ~30 s)
RESUME = False           # True: carry on from the last checkpoint in OUT
SAVE_MERGED = True       # the full model in 16-bit (about 9 GB): what Ollama imports
SAVE_GGUF = False        # GGUF files for llama.cpp (builds llama.cpp first: slow)

In [ ]:
import glob, json, os, sys, zipfile
from google.colab import drive
drive.mount("/content/drive")
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content/data")
DATA = os.path.dirname(glob.glob("/content/data/**/train.jsonl", recursive=True)[0])
sys.path.insert(0, os.path.join(DATA, "tools"))
import scenespec, score
train_rows = [json.loads(l) for l in open(os.path.join(DATA, "train.jsonl"), encoding="utf-8")]
val_rows = [json.loads(l) for l in open(os.path.join(DATA, "val.jsonl"), encoding="utf-8")]
names = json.load(open(os.path.join(DATA, "sprite_names.json")))
print(f"{len(train_rows):,} pictures to train on, {len(val_rows)} held back, {len(names)} sprite names")

In [ ]:
from unsloth import FastVisionModel
import torch
model, tokenizer = FastVisionModel.from_pretrained(MODEL, load_in_4bit=True, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,     # the vision part learns what your sprites look like
    finetune_language_layers=True,   # the language part learns the JSON
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_RANK, lora_alpha=LORA_RANK, lora_dropout=0, bias="none", random_state=3407,
)

In [ ]:
from PIL import Image

def messages(row, with_answer=True):
    # The picture first, then the instruction: the same order ask_model.py sends them in.
    image = Image.open(os.path.join(DATA, row["image"])).convert("RGB")
    out = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": row["prompt"]}]}]
    if with_answer:
        out.append({"role": "assistant", "content": [{"type": "text", "text": row["answer"]}]})
    return out

class Scenes(torch.utils.data.Dataset):
    # Opens each picture when it's needed, so thousands of pictures don't fill the memory.
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return {"messages": messages(self.rows[i])}

def ask(row):
    msgs = messages(row, with_answer=False)
    image = msgs[0]["content"][0]["image"]
    msgs[0]["content"][0] = {"type": "image"}
    prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = tokenizer(image, prompt, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=1500, do_sample=False, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def check(label):
    FastVisionModel.for_inference(model)
    results = []
    for row in val_rows[:CHECK_PICTURES]:
        answer = ask(row)
        try:
            guess = scenespec.parse_answer(answer)
            if scenespec.problems(guess, names):
                guess = None
        except ValueError:
            guess = None
        results.append(score.score(json.loads(row["answer"]), guess))
    print(label + ": " + score.describe(score.summary(results)))
    print("last answer:", answer[:300])

check("Before training")

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    data_collator=UnslothVisionDataCollator(
        model, tokenizer,
        resize="max",                          # keep the pictures' size (make_dataset keeps them at 640 or less)
        train_on_responses_only=True,          # learn the answers, not the instruction
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    ),
    train_dataset=Scenes(train_rows),
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,         # 8 pictures per step
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        bf16=torch.cuda.is_bf16_supported(),   # the free T4 has no bf16: fp16 there
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100, save_total_limit=2,    # checkpoints on Drive
        output_dir=os.path.join(OUT, "checkpoints"),
        seed=3407,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,                       # the longest example is about 1,100 tokens
    ),
)
trainer.train(resume_from_checkpoint=RESUME or None)

In [ ]:
check("After training")

In [ ]:
model.save_pretrained(os.path.join(OUT, "lora"))        # the add-on alone (small): keep it to train more later
tokenizer.save_pretrained(os.path.join(OUT, "lora"))
if SAVE_MERGED:
    model.save_pretrained_merged(os.path.join(OUT, "merged"), tokenizer, save_method="merged_16bit")
if SAVE_GGUF:
    model.save_pretrained_gguf(os.path.join(OUT, "gguf"), tokenizer, quantization_method="q4_k_m")
print("saved to", OUT)

## Using it on your PC

Download the `merged` folder (or the `gguf` files) from `scenes_model` in your Drive, then follow **Run your trained
model** in `VisionTools/README.md`: Ollama imports the merged folder, and llama.cpp's `llama-server` runs the GGUF
files. Then:

    python ask_model.py screenshot.png --sprites your_sprites --preview preview.png

and in Unity, *Tools → Vision → Build Scene From JSON*.